# scrollbar

> JavaScript generator for custom scrollbar interaction (drag thumb, click track, position sync).

In [ ]:
#| default_exp js.scrollbar

In [ ]:
#| export
from cjm_fasthtml_virtual_scrollbar.core.models import ScrollbarIds

## generate_scrollbar_js

Generates an IIFE that sets up all scrollbar interactions:

- **Thumb drag** — real-time visual feedback via direct DOM style updates, with debounced HTMX navigation
- **Track click** — jump to the clicked position
- **Position sync** — after any `htmx:afterSettle`, re-read the position from a hidden input and reposition the thumb
- **Deduplication** — `_lastNavIndex` prevents duplicate navigation requests during fast drag

The generated JS is consumer-agnostic. It reads position from a configurable hidden input element and POSTs to a configurable navigation URL.

In [ ]:
#| export
def generate_scrollbar_js(
    ids: ScrollbarIds,           # Scrollbar HTML IDs (track, thumb)
    position_input_id: str,      # ID of hidden input (kept for API compat, not used for position sync)
    nav_url: str,                # URL to POST target index to
    nav_param: str = "target_index",  # Parameter name for the index value
    on_interact: str = "",       # JS callback name, called on user interaction start (drag/click)
) -> str:  # JavaScript IIFE code fragment
    """Generate JS for scrollbar: thumb positioning from track data + drag/click interaction."""
    _interact_js = (
        f"if (typeof window['{on_interact}'] === 'function') window['{on_interact}']();"
        if on_interact else ""
    )
    return f"""
        // === Virtual Scrollbar ({ids.prefix}) ===
        (function() {{
            let _isDragging = false;
            let _lastNavIndex = -1;

            function _getTrack() {{ return document.getElementById('{ids.track}'); }}
            function _getThumb() {{ return document.getElementById('{ids.thumb}'); }}

            function _isCursorMode(track) {{
                // Cursor-based mode: explicit thumb_ratio and max_position on track.
                // In this mode, the scrollbar always shows a proportional thumb
                // regardless of visible vs total item count.
                return track.dataset.thumbRatio !== undefined && track.dataset.maxPosition !== undefined;
            }}

            function _getMaxPosition(track) {{
                // Read max position from data attribute (set by server render).
                // Defaults to totalItems - visibleCount if not present.
                if (track.dataset.maxPosition !== undefined) {{
                    return parseInt(track.dataset.maxPosition);
                }}
                const totalItems = parseInt(track.dataset.totalItems || '0');
                const visibleCount = parseInt(track.dataset.visibleCount || '1');
                return Math.max(0, totalItems - visibleCount);
            }}

            function _getThumbHeightPct(track) {{
                // Compute thumb height as percentage. Uses data-thumb-ratio if present,
                // else falls back to visibleCount / totalItems.
                const totalItems = parseInt(track.dataset.totalItems || '0');
                const visibleCount = parseInt(track.dataset.visibleCount || '1');
                if (totalItems <= 0) return 100;

                let pct;
                if (track.dataset.thumbRatio !== undefined) {{
                    pct = parseFloat(track.dataset.thumbRatio) * 100;
                }} else {{
                    pct = (visibleCount / totalItems) * 100;
                }}

                // Enforce minimum thumb height
                if (track.offsetHeight > 0) {{
                    const minPct = (24 / track.offsetHeight) * 100;
                    pct = Math.max(pct, minPct);
                }} else {{
                    pct = Math.max(pct, 4);
                }}
                return pct;
            }}

            function _positionThumbFromState() {{
                // Read position from track's own data-position attribute.
                // The scrollbar track is OOB-updated with fresh data on every navigation,
                // making position sync self-contained (no dependency on external hidden inputs).
                const track = _getTrack();
                const thumb = _getThumb();
                if (!track || !thumb) return;

                const totalItems = parseInt(track.dataset.totalItems || '0');
                const visibleCount = parseInt(track.dataset.visibleCount || '1');
                const position = parseInt(track.dataset.position || '0');

                // In default (window) mode, all items visible means nothing to scroll.
                // In cursor mode (explicit thumb_ratio + max_position), always show
                // proportional thumb as a position indicator.
                if (totalItems <= 0 || (!_isCursorMode(track) && visibleCount >= totalItems)) {{
                    thumb.style.top = '0%';
                    thumb.style.height = '100%';
                    return;
                }}

                const thumbHeightPct = _getThumbHeightPct(track);
                const maxPos = Math.max(1, _getMaxPosition(track));
                const thumbTopPct = (position / maxPos) * (100 - thumbHeightPct);

                thumb.style.top = thumbTopPct.toFixed(2) + '%';
                thumb.style.height = thumbHeightPct.toFixed(2) + '%';
            }}

            function _calcTargetIndex(pointerY) {{
                const track = _getTrack();
                if (!track) return 0;
                const rect = track.getBoundingClientRect();
                const maxPos = Math.max(0, _getMaxPosition(track));
                const relY = Math.max(0, Math.min(pointerY - rect.top, rect.height));
                return Math.round((relY / rect.height) * maxPos);
            }}

            function _updateThumbPosition(pointerY) {{
                // Direct DOM update for real-time visual feedback during drag
                const track = _getTrack();
                const thumb = _getThumb();
                if (!track || !thumb) return;
                const rect = track.getBoundingClientRect();
                const thumbHeight = thumb.offsetHeight;
                const maxTop = rect.height - thumbHeight;
                const relY = Math.max(0, Math.min(pointerY - rect.top - thumbHeight / 2, maxTop));
                thumb.style.top = relY + 'px';
            }}

            function _navToIndex(index) {{
                if (index === _lastNavIndex) return;  // Deduplicate
                _lastNavIndex = index;
                htmx.ajax('POST', '{nav_url}', {{
                    swap: 'none',
                    values: {{ {nav_param}: index }}
                }});
            }}

            // --- Document-level drag handlers ---

            function _onDragMove(evt) {{
                if (!_isDragging) return;
                evt.preventDefault();
                _updateThumbPosition(evt.clientY);
                const index = _calcTargetIndex(evt.clientY);
                _navToIndex(index);
            }}

            function _onDragEnd(evt) {{
                if (!_isDragging) return;
                _isDragging = false;
                _lastNavIndex = -1;
                document.removeEventListener('pointermove', _onDragMove);
                document.removeEventListener('pointerup', _onDragEnd);
                document.removeEventListener('pointercancel', _onDragEnd);
                const thumb = _getThumb();
                if (thumb) thumb.style.cursor = '';
                _setupScrollbar();
            }}

            function _setupScrollbar() {{
                const track = _getTrack();
                const thumb = _getThumb();
                if (!track || !thumb) return;

                if (track._scrollbarAbort) track._scrollbarAbort.abort();
                const controller = new AbortController();
                track._scrollbarAbort = controller;
                const opts = {{ signal: controller.signal }};

                // --- Thumb drag start ---
                thumb.addEventListener('pointerdown', function(evt) {{
                    if (evt.button !== 0) return;
                    evt.preventDefault();
                    evt.stopPropagation();
                    {_interact_js}
                    _isDragging = true;
                    _lastNavIndex = -1;
                    thumb.style.cursor = 'grabbing';
                    document.addEventListener('pointermove', _onDragMove);
                    document.addEventListener('pointerup', _onDragEnd);
                    document.addEventListener('pointercancel', _onDragEnd);
                }}, opts);

                // --- Track click (jump to position) ---
                track.addEventListener('pointerdown', function(evt) {{
                    if (evt.button !== 0) return;
                    const currentThumb = _getThumb();
                    if (currentThumb && (evt.target === currentThumb || currentThumb.contains(evt.target))) return;
                    evt.preventDefault();
                    {_interact_js}
                    const index = _calcTargetIndex(evt.clientY);
                    _lastNavIndex = -1;
                    _navToIndex(index);
                }}, opts);
            }}

            // Position thumb on initial load
            _positionThumbFromState();
            _setupScrollbar();

            // After each HTMX settle, re-position thumb from updated track data
            document.body.addEventListener('htmx:afterSettle', function() {{
                if (!_isDragging) {{
                    _positionThumbFromState();
                    _setupScrollbar();
                }}
            }});
        }})();
    """

## Tests

In [ ]:
ids = ScrollbarIds(prefix="demo")
js = generate_scrollbar_js(
    ids=ids,
    position_input_id="demo-position-input",
    nav_url="/demo/nav_to_index",
    nav_param="target_index",
)

# Verify key IDs are embedded
assert 'demo-scrollbar-track' in js
assert 'demo-scrollbar-thumb' in js

# Verify nav URL and param
assert '/demo/nav_to_index' in js
assert 'target_index: index' in js

# Verify IIFE wrapping
assert '(function()' in js
assert '})();' in js

# Verify data attribute names match render_scrollbar
assert 'dataset.totalItems' in js
assert 'dataset.visibleCount' in js
assert 'dataset.maxPosition' in js
assert 'dataset.thumbRatio' in js
assert 'dataset.position' in js  # Self-contained position sync
assert '_getMaxPosition' in js
assert '_getThumbHeightPct' in js
assert '_isCursorMode' in js  # Cursor-mode detection for always-visible thumb

# position_input_id is NOT referenced in the JS (kept for API compat only)
assert 'demo-position-input' not in js

# on_interact callback NOT present when not provided
assert 'on_interact' not in js.lower() or 'onInteract' not in js

In [ ]:
# Custom nav param — position_input_id kept for API compat but not in JS output
js2 = generate_scrollbar_js(
    ids=ScrollbarIds(prefix="cs0"),
    position_input_id="cs0-focused-index",
    nav_url="/card-stack/nav_to_index",
    nav_param="target_index",
)
assert 'cs0-scrollbar-track' in js2
assert '/card-stack/nav_to_index' in js2
assert 'dataset.position' in js2  # Reads from track, not hidden input

# on_interact callback present when provided
js3 = generate_scrollbar_js(
    ids=ScrollbarIds(prefix="sb1"),
    position_input_id="sb1-pos",
    nav_url="/sb1/nav_to_index",
    on_interact="onSb1Interact",
)
assert "onSb1Interact" in js3
assert "window['onSb1Interact']" in js3
# Callback invocation appears in both pointerdown handlers (thumb + track)
# Each handler has: typeof check + call = 2 occurrences per handler, 4 total
assert js3.count("window['onSb1Interact']") == 4

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()